# TETR.IO 風格 Tetris AI — Colab 一頁式流程

這份 notebook 把「環境設定 → 資料集檢查 → 模仿學習（IL）→ PPO 微調 → 監控與評估」全部放在一起，
**只要上傳這一個檔案**就能跑完整條流程。

執行的程式碼會自動從 GitHub 取得（公開 repo，不需要登入）：
`https://github.com/wallacechen0130/tetr_bot`

## 使用順序

1. **Cell 1** 環境 bootstrap：掛載 Drive、clone 專案、安裝缺少的套件
2. **Cell 2** 路徑與資料集檢查（自動挑選 Drive 上最新的資料集）
3. **Cell 3–4**（可選）在 Colab 產生資料集；通常在本機產生比較快
4. **Cell 5–6** 模仿學習：先用小樣本確認 loss 下降，再放大到全量
5. **Cell 7–8** PPO 微調：從 IL 權重熱啟動
6. **Cell 9–10** TensorBoard 監控與評估報告

> 提醒：Colab 免費版會斷線，checkpoint 都直接寫進 Google Drive，斷線後把同一個 cell 重跑即可續訓。

## Cell 1：環境 bootstrap

* 掛載 Google Drive（Colab 重啟後會自動重新掛載）
* 取得程式碼：已有 checkout 就 `git pull`，否則 `git clone`
* 只安裝缺少的套件（已裝過會跳過，重跑很快）

In [ ]:
import importlib.util
import os
import subprocess
import sys

# 1) 掛載 Drive（非 Colab 環境會自動跳過）
try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except Exception as exc:
    print('（非 Colab 環境或已掛載，略過）', exc)

# 2) 取得程式碼
PROJECT = '/content/tetrio-ai'
REPO_URL = 'https://github.com/wallacechen0130/tetr_bot.git'
if os.path.isdir(os.path.join(PROJECT, '.git')):
    print(subprocess.run(f'git -C {PROJECT} pull --ff-only', shell=True, capture_output=True, text=True).stdout)
elif not os.path.exists(os.path.join(PROJECT, 'requirements.txt')):
    subprocess.run(f'git clone --depth 1 {REPO_URL} {PROJECT}', shell=True, check=True)
os.chdir(PROJECT)

# 3) 只補裝缺少的套件
required = ['numpy', 'pandas', 'pyarrow', 'torch', 'stable_baselines3', 'sb3_contrib']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print('缺少套件，安裝 requirements-colab.txt：', missing)
    subprocess.run('pip -q install -r requirements-colab.txt', shell=True, check=True)
else:
    print('依賴已齊全')

import torch

print('python :', sys.version.split()[0])
print('torch  :', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu    :', torch.cuda.get_device_name(0))
print('repo   :', os.getcwd())
print(subprocess.run('git log --oneline -1', shell=True, capture_output=True, text=True).stdout.strip())

## Cell 2：路徑與資料集檢查

Drive 上的資料集放在 `MyDrive/tetrio-ai/datasets/<名稱>/`，
這裡會優先使用 `heuristic-v1`，沒有就用最新的那個。

In [ ]:
import glob
import json

os.environ.setdefault('TETRIO_AI_DRIVE', '/content/drive/MyDrive/tetrio-ai')
DRIVE_ROOT = os.environ['TETRIO_AI_DRIVE']
DATASETS_DIR = os.path.join(DRIVE_ROOT, 'datasets')

preferred = os.path.join(DATASETS_DIR, 'heuristic-v1')
found = sorted(path for path in glob.glob(os.path.join(DATASETS_DIR, '*')) if os.path.isdir(path))
DATA_ROOT = preferred if os.path.isdir(preferred) else (found[-1] if found else preferred)
IL_CKPT = os.path.join(DRIVE_ROOT, 'checkpoints', 'il')
PPO_CKPT = os.path.join(DRIVE_ROOT, 'checkpoints', 'ppo')
RUNS_DIR = os.path.join(DRIVE_ROOT, 'runs')
for path in (IL_CKPT, PPO_CKPT, RUNS_DIR):
    os.makedirs(path, exist_ok=True)

print('Drive      :', DRIVE_ROOT)
print('偵測到的資料集:', [os.path.basename(p) for p in found] or '（無）')
print('使用資料集  :', DATA_ROOT)
manifest_path = os.path.join(DATA_ROOT, 'manifest.json')
if os.path.exists(manifest_path):
    manifest = json.load(open(manifest_path, encoding='utf-8'))
    print('  樣本數:', manifest.get('num_samples'), '| shard 數:', manifest.get('num_shards'))
    shards = [s for s in manifest.get('shards', [])][:3]
    print('  前幾個 shard:', shards)
else:
    print('  尚未有 manifest.json（可用 Cell 3 產生，或從本機同步上來）')

# 先在這裡驗證資料集讀得到（跨平台路徑、Drive 是否同步完成）
if os.path.exists(manifest_path):
    from datasets.reader import DatasetReader
    probe = DatasetReader(DATA_ROOT, limit=1000)
    probe_arrays = probe.arrays()
    print('  讀取測試 OK：', probe_arrays['action'].shape, 'board', probe_arrays['board'].shape)

## Cell 3–4（可選）：產生資料集

資料產生是 CPU 密集工作，**本機跑比 Colab 快**（本機 8 workers 約 260 sample/s，147 萬筆約 1.5 小時）。
只有在 Drive 上還沒有資料集時才需要在這裡跑；已經有資料集就跳過這兩個 cell。

In [ ]:
# 只在沒有資料集時才產生（預設 20 局；要更大請自行調高 episodes）
if not os.path.exists(manifest_path):
    subprocess.run(
        f'python -m scripts.generate_dataset --out {DATA_ROOT} --episodes 40 --workers 4 --max-pieces 240',
        shell=True,
        check=True,
    )
else:
    print('已有資料集，略過產生：', DATA_ROOT)

## Cell 5：IL 小樣本驗證（先確認 loss 會下降）

用 2 萬筆、3 個 epoch 快速確認整條管線正常。正常的話 loss 會從 ~4.3 降到 ~3.5 左右
（80 類分類的隨機基線是 `ln(80) ≈ 4.38`），top-1 也會從 0.01 上升到 0.2 以上。

> 這裡直接用 Python API 呼叫（而不是 `!python -m ...`），因為在 notebook kernel 內執行時
> tqdm 會渲染成**原生進度條 widget**（含 ETA 與即時 val 指標）。用 CLI 也看得到進度條，
> 只是會以 `\r` 重畫同一行。

In [ ]:
from envs.config import load_yaml
from trainers.il_trainer import ILConfig, ILTrainer

il_config = ILConfig.from_dict(load_yaml('configs/il.yaml'))
il_config.data_root = DATA_ROOT
il_config.checkpoint_dir = IL_CKPT
il_config.network = 'small_cnn'   # CPU / Colab 都跑得動；想用 resnet 就改成 'resnet'
il_config.limit = 20000

small_trainer = ILTrainer(il_config)
small_result = small_trainer.fit(epochs=3)
print('完成：', {k: round(v, 4) for k, v in small_result['history'][-1].items()})

## Cell 6：IL 全量訓練

147 萬筆、`small_cnn` 在本機 CPU 約 3 分鐘/epoch；Colab T4 會更快。
`--network resnet`（預設）表達力更好但慢一些，PPO 也要用同一個網路名稱才吃得到權重。
early stopping 會在 top-1 連續 5 個 epoch 沒進步時停下。

In [ ]:
il_config.limit = None          # 全量（你的資料集是 147 萬筆）
full_trainer = ILTrainer(il_config)
full_result = full_trainer.fit(epochs=30)
print('best_top1:', round(full_result['best_top1'], 4))

## Cell 7：PPO 微調

從 IL 的 encoder 熱啟動。`--network` 要與 IL 相同，否則只會載到部分張量。
斷線後用 `--resume <checkpoint>` 續訓（`checkpoints/ppo/ppo_*_steps.zip`）。

In [ ]:
IL_BEST = os.path.join(IL_CKPT, 'best.pt')
print('IL checkpoint:', IL_BEST, os.path.exists(IL_BEST))

from trainers.ppo_trainer import PPOTrainer

ppo_config = load_yaml('configs/ppo.yaml')
ppo_config['env'].update({'n_envs': 4, 'vec': 'subproc', 'opponent_apm': 60.0})
ppo_config['model']['network'] = il_config.network   # 必須與 IL 相同，否則只會載到部分張量
ppo_config['train']['il_warm_start'] = IL_BEST
ppo_config['checkpoint']['dir'] = PPO_CKPT
ppo_config['logging']['tensorboard'] = RUNS_DIR

trainer = PPOTrainer(ppo_config, seed=12345)
smoke = trainer.run(total_timesteps=50_000)   # 先短跑確認 reward 有在動
print('IL 權重轉移:', smoke['transfer'])      # loaded/total 應該相等；不相等就是網路名稱不一致

In [ ]:
# 正式訓練（T4 約 1.5-3 小時）
ppo_config['env']['n_envs'] = 8
trainer = PPOTrainer(ppo_config, seed=12345)
trainer.run(total_timesteps=2_000_000)

# 斷線後續訓（checkpoint 都在 Drive）：
# trainer.run(total_timesteps=2_000_000, resume_from=f'{PPO_CKPT}/ppo_500000_steps.zip')

print(trainer.evaluate(episodes=3))

## Cell 8–9：監控與評估

TensorBoard 的 log 直接寫在 Drive 上，斷線重連後仍然看得到。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUNS_DIR}

In [ ]:
# 評估：啟發式教師基準 + 七級難度保真度
subprocess.run(
    f'python -m scripts.evaluate --agent heuristic --env-id Tetris40L-v0 --episodes 3 '
    f'--out-json {PPO_CKPT}/report.json --out-md {PPO_CKPT}/report.md',
    shell=True,
    check=True,
)
subprocess.run('python -m scripts.evaluate --difficulty-suite --max-steps 25', shell=True, check=True)

## 收尾：把訓練結果取回本機

模型與資料集都已經在 Google Drive 上，回到本機後執行：

```powershell
python -m scripts.sync_drive --list-drives                       # 先確認 Drive 路徑
$env:TETRIO_AI_DRIVE = "G:\我的雲端硬碟\tetrio-ai"                # 英文版：G:\My Drive\tetrio-ai
python -m scripts.sync_drive --direction from_drive --subdirs checkpoints,runs
```

## 疑難排解

| 症狀 | 處理 |
|---|---|
| `ModuleNotFoundError` | 重跑 Cell 1（會自動補裝缺少的套件） |
| 資料集 `FileNotFoundError` | Drive 還沒同步完，或 manifest 與檔案不符；重跑 Cell 2 看警告訊息 |
| loss 卡在 4.3 附近 | 確認是 `--limit` 小樣本還是全量；小樣本前幾個 epoch 本來就會在 4.0 上下 |
| PPO 的 `IL warm start` 顯示 0 個張量 | IL 與 PPO 的 `--network` 不一致，改用相同名稱 |
| Colab 斷線 | checkpoint 都在 Drive，重跑對應 cell 或加 `--resume` |